In [1]:
from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

%cd /content
!git clone https://{GITHUB_TOKEN}@github.com/Bassey-data/Auto-insurance-claim-frequency.git

%cd /content/Auto-insurance-claim-frequency
!git config user.email "basseysamuel404@gmail.com"
!git config user.name "Bassey-data"
!git remote set-url origin https://{GITHUB_TOKEN}@github.com/Bassey-data/Auto-insurance-claim-frequency.git
!git checkout feature/feature-engineering

print("Setup complete")

Mounted at /content/drive
/content
Cloning into 'Auto-insurance-claim-frequency'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 56 (delta 24), reused 30 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 464.41 KiB | 6.83 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Auto-insurance-claim-frequency
error: pathspec 'feature/feature-engineering' did not match any file(s) known to git
Setup complete


Import useful packages

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("Imports OK")

Imports OK


parse arff file

In [3]:
def parse_arff(path):
    columns = []
    data_start = None

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.lower().startswith("@attribute"):
            parts = stripped.split(maxsplit=2)
            columns.append(parts[1])
        elif stripped.lower().startswith("@data"):
            data_start = i + 1
            break

    data_lines = [l.strip() for l in lines[data_start:] if l.strip()]

    rows = []
    for line in data_lines:
        values = [v.strip().strip("'").strip('"') for v in line.split(",")]
        rows.append(values)

    return pd.DataFrame(rows, columns=columns)


parquet_path = "data/processed/freMTPL2freq.parquet"

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print("Loaded from parquet")
else:
    df = parse_arff("/content/drive/MyDrive/ACQsci.arff")
    df_converted = df.apply(pd.to_numeric, errors="ignore")
    for col in df_converted.select_dtypes(include="object").columns:
        df_converted[col] = df_converted[col].astype("category")
    df = df_converted
    os.makedirs("data/processed", exist_ok=True)
    df.to_parquet(parquet_path, index=False)
    print("Regenerated and saved parquet")

print(f"Shape: {df.shape}")

Regenerated and saved parquet
Shape: (678013, 12)


Cap outliers in VehAge column keep max age at 20 years

In [4]:

# VehAge: values above 20 are almost certainly data errors
df["VehAge"] = df["VehAge"].clip(upper=20)

# Exposure: a policy cannot be active for more than a full year
df["Exposure"] = df["Exposure"].clip(upper=1)

# ClaimNb: extreme values add noise without changing the risk
df["ClaimNb"] = df["ClaimNb"].clip(upper=4)

# Confirm if caps worked
print(f"VehAge max:   {df['VehAge'].max()}")
print(f"Exposure max: {df['Exposure'].max()}")
print(f"ClaimNb max:  {df['ClaimNb'].max()}")

VehAge max:   20
Exposure max: 1.0
ClaimNb max:  4


Area is an ordered risk classification (A=lowest, F=highest)

In [5]:

# Encode as integer to preserve the ordering
area_order = sorted(df["Area"].cat.categories)
area_map = {cat: i for i, cat in enumerate(area_order)}

df["Area_ord"] = df["Area"].map(area_map).astype(int)

print("Area encoding:", area_map)

Area encoding: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}


In [7]:
# Target and offset
target = "ClaimNb"
offset_col = "Exposure"

# Nominal categoricals - no natural order
nominal_cols = ["VehBrand", "VehGas", "Region"]

# Numeric features
numeric_cols = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density", "Area_ord"]

# Combined feature list
feature_cols = numeric_cols + nominal_cols

print("Features:", feature_cols)
print("Target:", target)
print("Offset:", offset_col)

Features: ['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density', 'Area_ord', 'VehBrand', 'VehGas', 'Region']
Target: ClaimNb
Offset: Exposure


In [11]:
# Stratify on whether a policy had any claim
# Ensures both sets have the same proportion of claimants
df["had_claim"] = (df[target] > 0).astype(int)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["had_claim"]
)

# Drop temporary stratification column
train_df = train_df.drop(columns="had_claim")
test_df = test_df.drop(columns="had_claim")

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"Train frequency: {train_df[target].sum() / train_df[offset_col].sum():.4f}")
print(f"Test frequency:  {test_df[target].sum() / test_df[offset_col].sum():.4f}")

Train: (542410, 13)
Test:  (135603, 13)
Train frequency: 0.1005
Test frequency:  0.1009


In [12]:
#  GLM feature set
# One-hot encode nominal categoricals, drop first to avoid dummy variable trap
train_glm = pd.get_dummies(train_df, columns=nominal_cols, drop_first=True)
test_glm = pd.get_dummies(test_df, columns=nominal_cols, drop_first=True)

# Align columns
train_glm, test_glm = train_glm.align(test_glm, join="left", axis=1, fill_value=0)

# Drop original Area column since Area_ord replaces it
train_glm = train_glm.drop(columns="Area")
test_glm = test_glm.drop(columns="Area")

# GBM feature set
# LightGBM handles categoricals natively so no one-hot needed
train_gbm = train_df.drop(columns="Area").copy()
test_gbm = test_df.drop(columns="Area").copy()

for col in nominal_cols:
    train_gbm[col] = train_gbm[col].astype("category")
    test_gbm[col] = test_gbm[col].astype("category")

print(f"GLM features: {len([c for c in train_glm.columns if c not in ['IDpol', target, offset_col]])}")
print(f"GBM features: {len([c for c in train_gbm.columns if c not in ['IDpol', target, offset_col]])}")

GLM features: 38
GBM features: 9


In [13]:
# Save model-ready datasets to processed folder
os.makedirs("data/processed", exist_ok=True)

train_glm.to_parquet("data/processed/train_glm.parquet", index=False)
test_glm.to_parquet("data/processed/test_glm.parquet", index=False)
train_gbm.to_parquet("data/processed/train_gbm.parquet", index=False)
test_gbm.to_parquet("data/processed/test_gbm.parquet", index=False)

print("Saved 4 files:")
print(f"  train_glm: {train_glm.shape}")
print(f"  test_glm:  {test_glm.shape}")
print(f"  train_gbm: {train_gbm.shape}")
print(f"  test_gbm:  {test_gbm.shape}")

Saved 4 files:
  train_glm: (542410, 41)
  test_glm:  (135603, 41)
  train_gbm: (542410, 12)
  test_gbm:  (135603, 12)
